In [ ]:
from utils import *

In [ ]:
root_directory = "/nfs/datz/olmo_models/new_processed_outputs/"

file_paths = traverse_directory(root_directory)
#main_files = [path for path in file_paths if ("subgraph" not in path and "test" not in path and "all_country_capital" in path)]

root_directory = "/nfs/datz/olmo_models/new_processed_outputs/"

all_files = [file for file in file_paths if "test" not in file]

models_data = read_all_files(all_files)

for model_name, relations in models_data.items():
    print(f"Model: {model_name}")
    for relation_name, data_entries in relations.items():
        print(f"  Relation: {relation_name}, Number of Entries: {len(data_entries)}")
        
sorted_models= sorted(
        models_data.keys(), 
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
)

In [ ]:
for x in models_data['main']:


In [ ]:
def compute_heatmaps(models_data, sorted_models, theta=0.1):
    """
    Computes aggregated heatmaps for a set of models.
    
    - The general and entity heatmaps are aggregated over all relations.
    - The relation-answer and answer-specific heatmaps are computed separately 
      for the two relation groups (LOC and NAME).
    
    Args:
        models_data (dict): Dictionary where each key is a model name and the value is a dict
                            mapping relation names to lists of entries.
        sorted_models (list): List of model names in the desired order.
        theta (float): Threshold for binarizing the heatmaps.
        
    Returns:
        tuple: Two dictionaries:
            - heatmaps_dict_LOC: Contains the overall general and entity heatmaps (shared) plus
              the LOC-specific relation-answer heatmaps.
            - heatmaps_dict_NAME: Contains the same overall general and entity heatmaps plus
              the NAME-specific relation-answer heatmaps.
    """
    # Define the two relation groups (in lower-case)
    relations = [
        "city_in_country", "company_hq", "country_capital_city",
        "food_from_country", "official_language", "plays_sport", "sights_in_city",
        "books_written", "company_ceo", "movie_directed"
    ]
    
    heatmaps_dict = {}
    
    for model_name in sorted_models:
        relations_data_model = models_data[model_name]
        print(model_name)
        
        # Initialize overall (general and entity) heatmaps and counters (for all relations)
        g_total = 0
        general_heatmap = np.zeros((32, 32), dtype=float)
        e_total = 0
        entity_heatmap = np.zeros((32, 32), dtype=float)
        
        # Initialize accumulators for relation-answer heatmaps for each group.
        relation_answer_heatmaps = {}
        relation_answer_with_specific = {}
        
        
        # Loop through each relation in the model's data.
        for relation, entries in relations_data_model.items():
            # Process only if the relation is in one of our groups.
            if relation not in relations:
                continue
            
            # Prepare accumulators for the relation-specific heatmaps.
            relation_answer_heatmap = np.zeros((32, 32), dtype=float)
            relation_answer_total = 0
            answer_specific_heatmaps = {}
            
            # Process each entry for this relation.
            for idx, entry in enumerate(entries):
                # Ensure subj_token_span is defined.
                if entry["subj_token_span"] is None:
                    entry["subj_token_span"] = np.arange(len(entry["subject_tokens"])).tolist()
                    
                # --- Update the overall general heatmap (averaged over all entries) ---
                # Use the average over the attention head contributions.
                general_heatmap += np.mean(entry['attnheads_contribution_maps'], axis=0)
                g_total += 1
                
                # --- Update the overall entity heatmap for subject tokens ---
                one_data_map = np.zeros((32, 32), dtype=float)
                for t in entry["subj_token_span"]:
                    one_data_map += entry['attnheads_contribution_maps'][t]
                entity_heatmap += one_data_map
                e_total += len(entry["subj_token_span"])
                
                # --- Update the overall entity and relation-answer heatmaps for answer tokens ---
                one_data_map = np.zeros((32, 32), dtype=float)
                for t in entry["answer_token_span"]:
                    one_data_map += entry['attnheads_contribution_maps'][t - 1]
                entity_heatmap += one_data_map
                e_total += len(entry["answer_token_span"])
                relation_answer_heatmap += one_data_map
                relation_answer_total += len(entry["answer_token_span"])
                
                # --- Store answer-specific heatmap for this entry ---
                answer_specific_heatmap = one_data_map / len(entry["answer_token_span"])
                answer_specific_heatmaps[idx] = answer_specific_heatmap > theta
            
            # Normalize the relation-answer heatmap for the current relation.
            if relation_answer_total > 0:
                relation_answer_heatmap /= relation_answer_total
            else:
                relation_answer_heatmap = np.zeros((32, 32), dtype=float)
            
            # Store the thresholded relation-answer heatmap and answer-specific maps
            # in the appropriate group dictionary.
            if relation in relations:
                relation_answer_heatmaps[relation] = relation_answer_heatmap > theta
                relation_answer_with_specific[relation] = answer_specific_heatmaps
            
        # Normalize the overall general and entity heatmaps over all entries.
        if g_total > 0:
            general_heatmap /= g_total
        if e_total > 0:
            entity_heatmap /= e_total
        
        # Apply thresholding.
        thresholded_general = general_heatmap > theta
        thresholded_entity = entity_heatmap > theta
        
        # Store the results for the current model.
        heatmaps_dict[model_name] = {
            'general_heatmap': thresholded_general,
            'entity_heatmap': thresholded_entity,
            'relation_answer_heatmaps': relation_answer_heatmaps,
            'relation_answer_with_specific': relation_answer_with_specific
        }
    
    return heatmaps_dict


heatmaps_dict = compute_heatmaps(models_data, sorted_models, theta=0.1)

In [ ]:
# def compute_heatmaps(models_data, sorted_models, theta=0.1):
    
#     relations = [
#             "city_in_country", "company_hq", "country_capital_city",
#             "food_from_country", "official_language", "plays_sport", "sights_in_city",
#             "books_written", "company_ceo", "movie_directed"
#         ]
        
#     heatmaps_dict = {}
    
#     for model_name in sorted_models:
#         relations_data_model = models_data[model_name]
#         print(model_name)
        
#         # Initialize dictionaries for per-relation heatmaps.
#         per_relation_general = {}
#         per_relation_entity = {}
#         relation_answer_heatmaps = {}
#         relation_answer_with_specific = {}
        
#         # Process each relation for the current model.
#         for relation, entries in relations_data_model.items():
#             if relation not in relations:
#                 continue
            
#             # Initialize per-relation accumulators.
#             general_heatmap_relation = np.zeros((32, 32), dtype=float)
#             g_total_relation = 0
#             entity_heatmap_relation = np.zeros((32, 32), dtype=float)
#             e_total_relation = 0
            
#             # Prepare accumulators for the relation-answer heatmap for this relation.
#             relation_answer_heatmap = np.zeros((32, 32), dtype=float)
#             relation_answer_total = 0
#             answer_specific_heatmaps = {}
            
#             # Process each entry for this relation.
#             for idx, entry in enumerate(entries):
#                 # Ensure subj_token_span is defined.
#                 if entry["subj_token_span"] is None:
#                     entry["subj_token_span"] = np.arange(len(entry["subject_tokens"])).tolist()
                    
#                 # --- Update the per-relation general heatmap ---
#                 general_heatmap_relation += np.mean(entry['attnheads_contribution_maps'], axis=0)
#                 g_total_relation += 1
                
#                 # --- Update the per-relation entity heatmap for subject tokens ---
#                 one_data_map = np.zeros((32, 32), dtype=float)
#                 for t in entry["subj_token_span"]:
#                     one_data_map += entry['attnheads_contribution_maps'][t]
#                 entity_heatmap_relation += one_data_map
#                 e_total_relation += len(entry["subj_token_span"])
                
#                 # --- Update the per-relation entity and relation-answer heatmaps for answer tokens ---
#                 one_data_map = np.zeros((32, 32), dtype=float)
#                 for t in entry["answer_token_span"]:
#                     one_data_map += entry['attnheads_contribution_maps'][t - 1]
#                 entity_heatmap_relation += one_data_map
#                 e_total_relation += len(entry["answer_token_span"])
#                 relation_answer_heatmap += one_data_map
#                 relation_answer_total += len(entry["answer_token_span"])
                
#                 # --- Store answer-specific heatmap for this entry ---
#                 answer_specific_heatmap = one_data_map / len(entry["answer_token_span"])
#                 answer_specific_heatmaps[idx] = answer_specific_heatmap > theta
            
#             # Normalize per-relation general and entity heatmaps.
#             if g_total_relation > 0:
#                 general_heatmap_relation /= g_total_relation
#             if e_total_relation > 0:
#                 entity_heatmap_relation /= e_total_relation
            
#             # Apply thresholding.
#             thresholded_general_relation = general_heatmap_relation > theta
#             thresholded_entity_relation = entity_heatmap_relation > theta
            
#             # Normalize the relation-answer heatmap.
#             if relation_answer_total > 0:
#                 relation_answer_heatmap /= relation_answer_total
#             else:
#                 relation_answer_heatmap = np.zeros((32, 32), dtype=float)
            
#             # Store the thresholded maps in the dictionaries.
#             per_relation_general[relation] = thresholded_general_relation
#             per_relation_entity[relation] = thresholded_entity_relation
#             relation_answer_heatmaps[relation] = relation_answer_heatmap > theta
#             relation_answer_with_specific[relation] = answer_specific_heatmaps
        
#         # Store the computed per-relation heatmaps for the current model.
#         heatmaps_dict[model_name] = {
#             'per_relation_general': per_relation_general,
#             'per_relation_entity': per_relation_entity,
#             'relation_answer_heatmaps': relation_answer_heatmaps,
#             'relation_answer_with_specific': relation_answer_with_specific
#         }
    
#     return heatmaps_dict

# # Example usage:
# heatmaps_dict = compute_heatmaps(models_data, sorted_models, theta=0.1)

In [ ]:
import numpy as np

def compute_subject_relation_answer_heatmaps(models_data, sorted_models, theta=0.1):
    all_relations = [
        "city_in_country", "company_hq", "country_capital_city",
        "food_from_country", "official_language", "plays_sport", "sights_in_city",
        "books_written", "company_ceo", "movie_directed"
    ]
    
    heatmaps_dict = {}
    
    for model_name in sorted_models:
        print(f"Processing model: {model_name}")
        relation_heatmaps = {}
        relations_data_model = models_data.get(model_name, {})
        
        for relation in all_relations:
            if relation not in relations_data_model:
                continue
            
            # Initialize accumulators for the heatmaps.
            general_heatmap = np.zeros((32, 32), dtype=float)
            subject_heatmap = np.zeros((32, 32), dtype=float)
            relation_heatmap = np.zeros((32, 32), dtype=float)
            answer_heatmap   = np.zeros((32, 32), dtype=float)
            
            # Counters for normalization
            general_total = 0
            subject_total = 0
            relation_total = 0
            answer_total = 0
            
            # Process each entry for the current relation.
            for entry in relations_data_model[relation]:
                # Retrieve token spans, with defaults.
                subj_span = entry.get("subj_token_span", [])
                ans_span  = entry.get("answer_token_span", [])
                
                # --- General tokens: use the mean over all token positions ---
                # This averages over the first axis of the attnheads_contribution_maps.
                general_map = np.mean(entry["attnheads_contribution_maps"], axis=0)
                general_heatmap += general_map
                general_total += 1
                
                # --- Subject tokens ---
                subj_map = np.zeros((32, 32), dtype=float)
                try:
                    for t in subj_span:
                        subj_map += entry["attnheads_contribution_maps"][t]
                except TypeError:
                    continue
                subject_heatmap += subj_map
                subject_total += len(subj_span)
                
                # --- Relation tokens: union of tokens before subject and between subject and answer ---
                relation_span = []
                if subj_span and ans_span:
                    min_subj = min(subj_span)
                    max_subj = max(subj_span)
                    min_ans  = min(ans_span)
                    
                    # Tokens before the subject (if subject doesn't start at index 0)
                    if min_subj > 0:
                        relation_span += list(range(0, min_subj))
                    
                    # Tokens strictly between the last subject token and the first answer token.
                    if max_subj < min_ans:
                        relation_span += list(range(max_subj + 1, min_ans))
                
                rel_map = np.zeros((32, 32), dtype=float)
                for t in relation_span:
                    rel_map += entry["attnheads_contribution_maps"][t]
                relation_heatmap += rel_map
                relation_total += len(relation_span)
                
                # --- Answer tokens ---
                ans_map = np.zeros((32, 32), dtype=float)
                for t in ans_span:
                    ans_map += entry["attnheads_contribution_maps"][t]
                answer_heatmap += ans_map
                answer_total += len(ans_span)
            
            # Normalize the heatmaps by the total number of tokens contributing.
            if general_total > 0:
                general_heatmap /= general_total
            if subject_total > 0:
                subject_heatmap /= subject_total
            if relation_total > 0:
                relation_heatmap /= relation_total
            if answer_total > 0:
                answer_heatmap /= answer_total
            
            # Apply thresholding if binary maps are desired.
            general_heatmap = (general_heatmap > theta)
            subject_heatmap = (subject_heatmap > theta)
            relation_heatmap = (relation_heatmap > theta)
            answer_heatmap = (answer_heatmap > theta)
            
            # Store the computed heatmaps for the current relation.
            relation_heatmaps[relation] = {
                'general_heatmap':  general_heatmap,
                'subject_heatmap':  subject_heatmap,
                'relation_heatmap': relation_heatmap,
                'answer_heatmap':   answer_heatmap
            }
        
        # Save the results for the current model.
        heatmaps_dict[model_name] = relation_heatmaps
    
    return heatmaps_dict

heatmaps_dict = compute_subject_relation_answer_heatmaps(models_data, sorted_models, theta=0.1)

In [ ]:
heatmaps_dict

In [ ]:
def calculate_proper_heads(heatmaps_dict):
    """
    Calculates proper entity heads, proper relation answer heads, 
    and proper answer specific heads for each model in heatmaps_dict.

    Args:
        heatmaps_dict (dict): A dictionary where each key is a model name and 
                              the value is a dictionary with heatmaps.

    Returns:
        dict: A dictionary containing proper heads for each model.
    """
    proper_heads_dict = {}

    for model_name, heatmaps in heatmaps_dict.items():
        # Extract heatmaps
        general_heads = heatmaps['general_heatmap']
        entity_heads = heatmaps['entity_heatmap']
        relation_answer_heads = heatmaps['relation_answer_heatmaps']
        answer_specific_heads = heatmaps['relation_answer_with_specific']

        # Calculate proper heads
        proper_entity_heads = np.logical_and(entity_heads, np.logical_not(general_heads))
        
        # Initialize dictionaries to store results for relations and specific answers
        proper_relation_answer_heads = {}
        proper_answer_specific_heads = {}

        # Calculate proper relation answer heads per relation
        for relation, relation_heads in relation_answer_heads.items():
            proper_relation_answer_heads[relation] = np.logical_and(
                relation_heads,
                np.logical_not(np.logical_or(proper_entity_heads, general_heads))
            )

        # Calculate proper answer specific heads per relation and specific answer
        for relation, specific_answers in answer_specific_heads.items():
            proper_answer_specific_heads[relation] = {}
            for answer_id, specific_heads in specific_answers.items():
                proper_answer_specific_heads[relation][answer_id] = np.logical_and(
                    specific_heads,
                    np.logical_not(
                        np.logical_or(
                            proper_relation_answer_heads[relation],
                            np.logical_or(proper_entity_heads, general_heads)
                        )
                    )
                )

        # Store results in the output dictionary
        proper_heads_dict[model_name] = {
            'proper_general_heads': general_heads,
            'proper_entity_heads': proper_entity_heads,
            'proper_relation_answer_heads': proper_relation_answer_heads,
            'proper_answer_specific_heads': proper_answer_specific_heads
        }

    return proper_heads_dict

# Now, assuming you have built two separate dictionaries:
#   heatmaps_dict_LOC and heatmaps_dict_NAME
# you can calculate the proper heads for each group as follows:

proper_heads = calculate_proper_heads(heatmaps_dict)

In [ ]:
# def calculate_proper_heads(heatmaps_dict):
#     """
#     Calculates proper entity heads, proper relation answer heads, 
#     and proper answer specific heads for each model in heatmaps_dict.
#     This version assumes that the general and entity heatmaps are computed 
#     on a per-relation basis (stored in dictionaries).

#     Args:
#         heatmaps_dict (dict): A dictionary where each key is a model name and 
#                               the value is a dictionary with per-relation heatmaps.
#                               Expected keys:
#                                 - 'per_relation_general'
#                                 - 'per_relation_entity'
#                                 - 'relation_answer_heatmaps'
#                                 - 'relation_answer_with_specific'
#     Returns:
#         dict: A dictionary containing proper heads for each model.
#     """
#     proper_heads_dict = {}

#     for model_name, heatmaps in heatmaps_dict.items():
#         # Extract per-relation heatmaps
#         per_relation_general = heatmaps['per_relation_general']
#         per_relation_entity = heatmaps['per_relation_entity']
#         relation_answer_heads = heatmaps['relation_answer_heatmaps']
#         answer_specific_heads = heatmaps['relation_answer_with_specific']

#         # Initialize dictionaries to store the proper heads per relation
#         proper_general_heads = {}
#         proper_entity_heads = {}
#         proper_relation_answer_heads = {}
#         proper_answer_specific_heads = {}

#         # Iterate over each relation
#         for relation in per_relation_general:
#             general_heads = per_relation_general[relation]
#             entity_heads = per_relation_entity[relation]

#             # Compute proper entity heads for this relation:
#             # heads that are entity-specific but not general.
#             proper_entity_heads[relation] = np.logical_and(entity_heads, np.logical_not(general_heads))

#             # For proper general heads, we can simply use the computed general heads.
#             proper_general_heads[relation] = general_heads

#             # Calculate proper relation answer heads for this relation:
#             # those that are marked in the relation-answer heatmap
#             # but not already captured by proper entity or general heads.
#             proper_relation_answer_heads[relation] = np.logical_and(
#                 relation_answer_heads[relation],
#                 np.logical_not(np.logical_or(proper_entity_heads[relation], general_heads))
#             )

#             # Calculate proper answer specific heads per relation.
#             proper_answer_specific_heads[relation] = {}
#             for answer_id, specific_heads in answer_specific_heads[relation].items():
#                 proper_answer_specific_heads[relation][answer_id] = np.logical_and(
#                     specific_heads,
#                     np.logical_not(
#                         np.logical_or(
#                             proper_relation_answer_heads[relation],
#                             np.logical_or(proper_entity_heads[relation], general_heads)
#                         )
#                     )
#                 )

#         # Store the computed proper heads for the current model.
#         proper_heads_dict[model_name] = {
#             'proper_general_heads': proper_general_heads,
#             'proper_entity_heads': proper_entity_heads,
#             'proper_relation_answer_heads': proper_relation_answer_heads,
#             'proper_answer_specific_heads': proper_answer_specific_heads
#         }

#     return proper_heads_dict

# # Example usage:
# proper_heads = calculate_proper_heads(heatmaps_dict)

In [ ]:
# def calculate_proper_heads(heatmaps_dict):
#     """
#     Calculates proper subject, proper relation, and proper answer heads for each model and relation.
    
#     Assumes each model in heatmaps_dict contains for each relation:
#       - 'general_heatmap'
#       - 'subject_heatmap'
#       - 'relation_heatmap'
#       - 'answer_heatmap'
      
#     The proper heads are defined as:
#       - proper_general_heads = general_heatmap
#       - proper_subject_heads = subject_heatmap AND NOT general_heatmap
#       - proper_relation_heads = relation_heatmap AND NOT (subject_heatmap OR general_heatmap)
#       - proper_answer_heads = answer_heatmap AND NOT (subject_heatmap OR relation_heatmap OR general_heatmap)
    
#     Args:
#         heatmaps_dict (dict): A dictionary where each key is a model name and the value is a dictionary 
#                               of relations with their corresponding heatmaps.
    
#     Returns:
#         dict: A dictionary containing the proper heads for each model and each relation.
#     """
#     proper_heads_dict = {}
    
#     for model_name, relations in heatmaps_dict.items():
#         proper_heads_model = {}
#         for relation, heads in relations.items():
#             # Extract heatmaps for this relation.
#             general_heads = heads['general_heatmap']
#             subject_heads = heads['subject_heatmap']
#             relation_heads = heads['relation_heatmap']
#             answer_heads = heads['answer_heatmap']
            
#             # Compute proper heads.
#             proper_subject_heads = np.logical_and(subject_heads, np.logical_not(general_heads))
#             proper_relation_heads = np.logical_and(relation_heads,
#                                                    np.logical_not(np.logical_or(subject_heads, general_heads)))
#             proper_answer_heads = np.logical_and(answer_heads,
#                                                  np.logical_not(np.logical_or(np.logical_or(subject_heads, relation_heads),
#                                                                               general_heads)))
            
#             # Store proper heads for the current relation.
#             proper_heads_model[relation] = {
#                 'proper_general_heads': general_heads,
#                 'proper_subject_heads': proper_subject_heads,
#                 'proper_relation_heads': proper_relation_heads,
#                 'proper_answer_heads': proper_answer_heads
#             }
        
#         proper_heads_dict[model_name] = proper_heads_model
    
#     return proper_heads_dict

# proper_heads = calculate_proper_heads(heatmaps_dict)

In [ ]:
proper_heads

In [ ]:
def calculate_consistency_proper_heads(heatmaps_dict, relation_name, fact_idx):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        #if model_name == 'main':
        #    continue

        # Store IoU results for this model
        model_ious = {}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "proper_relation_answer_heads":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]
            elif heatmap_type == "proper_answer_specific_heads":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_heatmap = model_data[heatmap_type][relation_name][fact_idx]
            elif heatmap_type == "proper_entity_heads":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            elif heatmap_type == "proper_general_heads":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            else:
                continue

            # Calculate IoU
            intersection = np.logical_and(main_heatmap, model_heatmap)
            union = np.logical_or(main_heatmap, model_heatmap)

            area_of_intersection = np.sum(intersection)
            area_of_union = np.sum(union)

            # Avoid division by zero
            iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

            # Store IoU result and statistics
            model_ious[heatmap_type] = {
                'iou': iou,
                'head_count': model_heatmap.sum(),
                'intersection': area_of_intersection,
                'union': area_of_union
            }

        # Add this model's IoU results to the main dictionary
        iou_results[model_name] = model_ious

    return iou_results

In [ ]:
# def calculate_consistency_proper_heads(heatmaps_dict, relation_name, fact_idx):
#     iou_results = {}

#     # Retrieve the 'main' model's proper heads heatmaps
#     main_heatmaps = heatmaps_dict['main']

#     # Iterate through each model in the dictionary
#     for model_name, model_data in heatmaps_dict.items():
#         # Optionally, you could exclude 'main' if desired
#         # if model_name == 'main':
#         #     continue

#         # Dictionary to store IoU results for this model
#         model_ious = {}

#         for heatmap_type in main_heatmaps.keys():
#             if heatmap_type == "proper_relation_answer_heads":
#                 # Index by relation_name for relation-answer heatmaps
#                 main_heatmap = main_heatmaps[heatmap_type][relation_name]
#                 model_heatmap = model_data[heatmap_type][relation_name]
#             elif heatmap_type == "proper_answer_specific_heads":
#                 # Index by relation_name and fact_idx for answer-specific heatmaps
#                 main_heatmap = main_heatmaps[heatmap_type][relation_name][fact_idx]
#                 model_heatmap = model_data[heatmap_type][relation_name][fact_idx]
#             elif heatmap_type == "proper_entity_heads":
#                 # Now also index per relation
#                 main_heatmap = main_heatmaps[heatmap_type][relation_name]
#                 model_heatmap = model_data[heatmap_type][relation_name]
#             elif heatmap_type == "proper_general_heads":
#                 # Now also index per relation
#                 main_heatmap = main_heatmaps[heatmap_type][relation_name]
#                 model_heatmap = model_data[heatmap_type][relation_name]
#             else:
#                 continue

#             # Calculate Intersection over Union (IoU)
#             intersection = np.logical_and(main_heatmap, model_heatmap)
#             union = np.logical_or(main_heatmap, model_heatmap)
#             area_of_intersection = np.sum(intersection)
#             area_of_union = np.sum(union)
#             iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

#             # Store IoU result along with some statistics
#             model_ious[heatmap_type] = {
#                 'iou': iou,
#                 'head_count': np.sum(model_heatmap),
#                 'intersection': area_of_intersection,
#                 'union': area_of_union
#             }

#         iou_results[model_name] = model_ious

#     return iou_results

In [ ]:
# def calculate_consistency_proper_heads(heatmaps_dict, relation_name):
#     """
#     Calculates the Intersection over Union (IoU) for proper heads of a given relation 
#     across different models. For each model, the IoU is computed for the following head types:
#         - proper_general_heads
#         - proper_subject_heads
#         - proper_relation_heads
#         - proper_answer_heads

#     Assumes that the heatmaps_dict is structured as follows:
#         {
#             model_name: {
#                 relation_name: {
#                     'proper_general_heads': np.array(...),
#                     'proper_subject_heads': np.array(...),
#                     'proper_relation_heads': np.array(...),
#                     'proper_answer_heads': np.array(...)
#                 },
#                 ...
#             },
#             ...
#         }

#     Args:
#         heatmaps_dict (dict): A dictionary containing proper heads for each model and relation.
#         relation_name (str): The name of the relation to evaluate.

#     Returns:
#         dict: A dictionary containing IoU results for each model.
#     """
#     iou_results = {}

#     # Retrieve proper heads for the specified relation from the 'main' model.
#     main_relation_heads = heatmaps_dict['main'][relation_name]

#     # Iterate through each model.
#     for model_name, model_data in heatmaps_dict.items():
#         # Each model's data for the given relation.
#         model_relation_heads = model_data[relation_name]
#         model_ious = {}

#         # Compute IoU for each head type.
#         for head_type, main_heatmap in main_relation_heads.items():
#             model_heatmap = model_relation_heads[head_type]
#             intersection = np.logical_and(main_heatmap, model_heatmap)
#             union = np.logical_or(main_heatmap, model_heatmap)

#             area_of_intersection = np.sum(intersection)
#             area_of_union = np.sum(union)
#             iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

#             model_ious[head_type] = {
#                 'iou': iou,
#                 'head_count': np.sum(model_heatmap),
#                 'intersection': area_of_intersection,
#                 'union': area_of_union
#             }
#         iou_results[model_name] = model_ious

#     return iou_results

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt

# Assuming these helper functions are defined:
def extract_step_number(model_name):
    """
    Extracts the step number from the model name using regex.
    'main' is assigned an infinite step so that it comes last.
    """
    if model_name == 'main':
        return float('inf')
    match = re.search(r'\d+', model_name)
    return int(match.group()) if match else float('inf')

def abbreviate_model_name(model_name, index):
    """
    Converts a model name like 'step10000-tokens41B' into a compact label 'S{index}-41B'.
    
    For example, if index is 1 and model_name is 'step10000-tokens41B', then the label
    will be 'S1-41B'. For 'main', we simply return 'Main'.
    """
    if model_name == 'main':
        return "Main"
    token_match = re.search(r'tokens(\d+B)', model_name)
    token_part = token_match.group(1) if token_match else ""
    return f"S{index}-{token_part}"


def plot_aggregated_proper_heads_iou(iou_results, relation_name, output_dir):
    # ------------------------------
    # Prepare data and abbreviate model names.
    # ------------------------------
    # Sort model names (with 'main' coming last)
    sorted_model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    
    # Create abbreviated model names using the helper function.
    abbreviated_model_names = [
        abbreviate_model_name(model, i) for i, model in enumerate(sorted_model_names, start=1)
    ]
    
    # For the IoU plot, exclude the 'main' snapshot if it is the last element.
    if sorted_model_names[-1] == 'main':
        sorted_model_names_iou = sorted_model_names[:-1]
        abbreviated_model_names_iou = abbreviated_model_names[:-1]
    else:
        sorted_model_names_iou = sorted_model_names
        abbreviated_model_names_iou = abbreviated_model_names

    # Initialize lists for IoU metrics.
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []
    
    # Initialize lists for head counts.
    general_head_count = []
    entity_head_count = []
    answer_all_head_count = []
    answer_head_counts = []
    
    # Populate IoU lists (excluding main)
    for model_name in sorted_model_names_iou:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])
    
    # Populate head count lists (including main)
    for model_name in sorted_model_names:
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_head_counts.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])
    
    # Create directories if they do not exist.
    iou_dir = os.path.join(output_dir, "iou")
    count_dir = os.path.join(output_dir, "count")
    os.makedirs(iou_dir, exist_ok=True)
    os.makedirs(count_dir, exist_ok=True)
    
    # ------------------------------
    # Plot and save the Aggregated IoU figure.
    # ------------------------------
    plt.figure(figsize=(20, 6))
    plt.plot(abbreviated_model_names_iou, general_ious, label="Proper General Heads IoU (Aggregated)", marker='o', linestyle='-')
    plt.plot(abbreviated_model_names_iou, entity_ious, label="Proper Entity Heads IoU (Aggregated)", marker='s', linestyle='--')
    plt.plot(abbreviated_model_names_iou, answer_all_ious, label="Proper Relation Answer Heads IoU (Aggregated)", marker='x', linestyle='-.')
    plt.plot(abbreviated_model_names_iou, answer_ious, label="Proper Answer Specific Heads IoU (Aggregated)", marker='^', linestyle='-.')

    plt.xticks(rotation=45, ha='right')
    plt.xlabel("Model Snapshots")
    plt.ylabel("Aggregated IoU")
    plt.title(f"Aggregated Proper Heads IoU Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    iou_filename = os.path.join(iou_dir, f"{relation_name}_heads_iou.pdf")
    plt.savefig(iou_filename, dpi=500, bbox_inches='tight', format="pdf")
    print(f"IoU plot saved to {iou_filename}")
    plt.show()
    plt.close()
    
    # ------------------------------
    # Plot and save the Aggregated Head Count figure.
    # ------------------------------
    plt.figure(figsize=(20, 6))
    plt.plot(abbreviated_model_names, general_head_count, label="Proper General Heads Count (Aggregated)", marker='o', linestyle='-')
    plt.plot(abbreviated_model_names, entity_head_count, label="Proper Entity Heads Count (Aggregated)", marker='s', linestyle='--')
    plt.plot(abbreviated_model_names, answer_all_head_count, label="Proper Relation Answer Heads Count (Aggregated)", marker='x', linestyle='-.')
    plt.plot(abbreviated_model_names, answer_head_counts, label="Proper Answer Specific Heads Count (Aggregated)", marker='^', linestyle='-.')

    plt.xticks(rotation=45, ha='right')
    plt.xlabel("Model Snapshots")
    plt.ylabel("Aggregated Count")
    plt.title(f"Aggregated Proper Heads Count Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    count_filename = os.path.join(count_dir, f"{relation_name}_heads_count.pdf")
    plt.savefig(count_filename, dpi=500, bbox_inches='tight', format="pdf")
    print(f"Head count plot saved to {count_filename}")
    plt.show()
    plt.close()

In [ ]:
# import os
# import re
# import numpy as np
# import matplotlib.pyplot as plt


# def extract_step_number(model_name):
#     """
#     Extracts the step number from the model name using regex.
#     'main' is assigned an infinite step so that it comes last.
#     """
#     if model_name == 'main':
#         return float('inf')
#     match = re.search(r'\d+', model_name)
#     return int(match.group()) if match else float('inf')

# def abbreviate_model_name(model_name, index):
#     """
#     Converts a model name like 'step10000-tokens41B' into a compact label 'S{index}-41B'.
    
#     For example, if index is 1 and model_name is 'step10000-tokens41B', then the label
#     will be 'S1-41B'. For 'main', we simply return 'Main'.
#     """
#     if model_name == 'main':
#         return "Main"
#     token_match = re.search(r'tokens(\d+B)', model_name)
#     token_part = token_match.group(1) if token_match else ""
#     return f"S{index}-{token_part}"



# def plot_aggregated_proper_heads_iou(iou_results, relation_name, output_dir):
#     # ------------------------------
#     # Prepare data and abbreviate model names.
#     # ------------------------------
#     # Sort model names (with 'main' coming last)
#     sorted_model_names = sorted(
#         iou_results.keys(),
#         key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
#     )
    
#     # Create abbreviated model names using the helper function.
#     abbreviated_model_names = [
#         abbreviate_model_name(model, i) for i, model in enumerate(sorted_model_names, start=1)
#     ]
    
#     # For the IoU plot, exclude the 'main' snapshot if it is the last element.
#     if sorted_model_names[-1] == 'main':
#         sorted_model_names_iou = sorted_model_names[:-1]
#         abbreviated_model_names_iou = abbreviated_model_names[:-1]
#     else:
#         sorted_model_names_iou = sorted_model_names
#         abbreviated_model_names_iou = abbreviated_model_names

#     # ------------------------------
#     # Initialize lists for IoU metrics and head counts.
#     # ------------------------------
#     general_ious = []
#     entity_ious = []
#     answer_all_ious = []
#     answer_ious = []
    
#     general_head_count = []
#     entity_head_count = []
#     answer_all_head_count = []
#     answer_head_counts = []
    
#     # ------------------------------
#     # Populate IoU lists (excluding 'main').
#     # For each model, index the proper head metrics by relation_name.
#     # For answer-specific heads, average over all fact indices.
#     # ------------------------------
#     for model_name in sorted_model_names_iou:
#         general_ious.append(iou_results[model_name]['proper_general_heads'][relation_name]['iou'])
#         entity_ious.append(iou_results[model_name]['proper_entity_heads'][relation_name]['iou'])
#         answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads'][relation_name]['iou'])
        
#         answer_specific_dict = iou_results[model_name]['proper_answer_specific_heads'][relation_name]
#         iou_vals = [metrics['iou'] for metrics in answer_specific_dict.values()]
#         avg_iou = np.mean(iou_vals) if iou_vals else 0
#         answer_ious.append(avg_iou)
    
#     # ------------------------------
#     # Populate head count lists (including 'main').
#     # ------------------------------
#     for model_name in sorted_model_names:
#         general_head_count.append(iou_results[model_name]['proper_general_heads'][relation_name]['head_count'])
#         entity_head_count.append(iou_results[model_name]['proper_entity_heads'][relation_name]['head_count'])
#         answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads'][relation_name]['head_count'])
        
#         answer_specific_dict = iou_results[model_name]['proper_answer_specific_heads'][relation_name]
#         counts = [metrics['head_count'] for metrics in answer_specific_dict.values()]
#         avg_count = np.mean(counts) if counts else 0
#         answer_head_counts.append(avg_count)
    
#     # ------------------------------
#     # Create directories if they do not exist.
#     # ------------------------------
#     iou_dir = os.path.join(output_dir, "iou")
#     count_dir = os.path.join(output_dir, "count")
#     os.makedirs(iou_dir, exist_ok=True)
#     os.makedirs(count_dir, exist_ok=True)
    
#     # ------------------------------
#     # Plot and save the Aggregated IoU figure.
#     # ------------------------------
#     plt.figure(figsize=(20, 6))
#     plt.plot(abbreviated_model_names_iou, general_ious, label="Proper General Heads IoU (Aggregated)", marker='o', linestyle='-')
#     plt.plot(abbreviated_model_names_iou, entity_ious, label="Proper Entity Heads IoU (Aggregated)", marker='s', linestyle='--')
#     plt.plot(abbreviated_model_names_iou, answer_all_ious, label="Proper Relation Answer Heads IoU (Aggregated)", marker='x', linestyle='-.')
#     plt.plot(abbreviated_model_names_iou, answer_ious, label="Proper Answer Specific Heads IoU (Aggregated)", marker='^', linestyle='-.')
    
#     plt.xticks(rotation=45, ha='right')
#     plt.xlabel("Model Snapshots")
#     plt.ylabel("Aggregated IoU")
#     plt.title(f"Aggregated Proper Heads IoU Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
    
#     iou_filename = os.path.join(iou_dir, f"{relation_name}_heads_iou.pdf")
#     plt.savefig(iou_filename, dpi=500, bbox_inches='tight', format="pdf")
#     print(f"IoU plot saved to {iou_filename}")
#     plt.show()
#     plt.close()
    
#     # ------------------------------
#     # Plot and save the Aggregated Head Count figure.
#     # ------------------------------
#     plt.figure(figsize=(20, 6))
#     plt.plot(abbreviated_model_names, general_head_count, label="Proper General Heads Count (Aggregated)", marker='o', linestyle='-')
#     plt.plot(abbreviated_model_names, entity_head_count, label="Proper Entity Heads Count (Aggregated)", marker='s', linestyle='--')
#     plt.plot(abbreviated_model_names, answer_all_head_count, label="Proper Relation Answer Heads Count (Aggregated)", marker='x', linestyle='-.')
#     plt.plot(abbreviated_model_names, answer_head_counts, label="Proper Answer Specific Heads Count (Aggregated)", marker='^', linestyle='-.')
    
#     plt.xticks(rotation=45, ha='right')
#     plt.xlabel("Model Snapshots")
#     plt.ylabel("Aggregated Count")
#     plt.title(f"Aggregated Proper Heads Count Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
    
#     count_filename = os.path.join(count_dir, f"{relation_name}_heads_count.pdf")
#     plt.savefig(count_filename, dpi=500, bbox_inches='tight', format="pdf")
#     print(f"Head count plot saved to {count_filename}")
#     plt.show()
#     plt.close()

In [ ]:
# import os
# import re
# import numpy as np
# import matplotlib.pyplot as plt

# def extract_step_number(model_name):
#     """
#     Extracts the step number from the model name using regex.
#     'main' is assigned an infinite step so that it comes last.
#     """
#     if model_name == 'main':
#         return float('inf')
#     match = re.search(r'\d+', model_name)
#     return int(match.group()) if match else float('inf')

# def abbreviate_model_name(model_name, index):
#     """
#     Converts a model name like 'step10000-tokens41B' into a compact label 'S{index}-41B'.
    
#     For example, if index is 1 and model_name is 'step10000-tokens41B', then the label
#     will be 'S1-41B'. For 'main', we simply return 'Main'.
#     """
#     if model_name == 'main':
#         return "Main"
#     token_match = re.search(r'tokens(\d+B)', model_name)
#     token_part = token_match.group(1) if token_match else ""
#     return f"S{index}-{token_part}"

# def plot_aggregated_proper_heads_iou(iou_results, relation_name, output_dir):
#     # ------------------------------
#     # Prepare data and abbreviate model names.
#     # ------------------------------
#     # Sort model names (with 'main' coming last)
#     sorted_model_names = sorted(
#         iou_results.keys(),
#         key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
#     )
    
#     # Create abbreviated model names using the helper function.
#     abbreviated_model_names = [
#         abbreviate_model_name(model, i) for i, model in enumerate(sorted_model_names, start=1)
#     ]
    
#     # For the IoU plot, exclude the 'main' snapshot if it is the last element.
#     if sorted_model_names[-1] == 'main':
#         sorted_model_names_iou = sorted_model_names[:-1]
#         abbreviated_model_names_iou = abbreviated_model_names[:-1]
#     else:
#         sorted_model_names_iou = sorted_model_names
#         abbreviated_model_names_iou = abbreviated_model_names

#     # ------------------------------
#     # Initialize lists for IoU metrics and head counts.
#     # ------------------------------
#     # IoU metrics (excluding 'main')
#     general_ious = []
#     subject_ious = []
#     relation_ious = []
#     answer_ious = []
    
#     # Head count metrics (including 'main')
#     general_head_count = []
#     subject_head_count = []
#     relation_head_count = []
#     answer_head_count = []
    
#     # Populate IoU lists (excluding main)
#     for model_name in sorted_model_names_iou:
#         general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
#         subject_ious.append(iou_results[model_name]['proper_subject_heads']['iou'])
#         relation_ious.append(iou_results[model_name]['proper_relation_heads']['iou'])
#         answer_ious.append(iou_results[model_name]['proper_answer_heads']['iou'])
    
#     # Populate head count lists (including main)
#     for model_name in sorted_model_names:
#         general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
#         subject_head_count.append(iou_results[model_name]['proper_subject_heads']['head_count'])
#         relation_head_count.append(iou_results[model_name]['proper_relation_heads']['head_count'])
#         answer_head_count.append(iou_results[model_name]['proper_answer_heads']['head_count'])
    
#     # Create directories if they do not exist.
#     iou_dir = os.path.join(output_dir, "iou")
#     count_dir = os.path.join(output_dir, "count")
#     os.makedirs(iou_dir, exist_ok=True)
#     os.makedirs(count_dir, exist_ok=True)
    
#     # ------------------------------
#     # Plot and save the Aggregated IoU figure.
#     # ------------------------------
#     plt.figure(figsize=(20, 6))
#     plt.plot(abbreviated_model_names_iou, general_ious, label="Proper General Heads IoU (Aggregated)", marker='o', linestyle='-')
#     plt.plot(abbreviated_model_names_iou, subject_ious, label="Proper Subject Heads IoU (Aggregated)", marker='s', linestyle='--')
#     plt.plot(abbreviated_model_names_iou, relation_ious, label="Proper Relation Heads IoU (Aggregated)", marker='x', linestyle='-.')
#     plt.plot(abbreviated_model_names_iou, answer_ious, label="Proper Answer Heads IoU (Aggregated)", marker='^', linestyle='-.')

#     plt.xticks(rotation=45, ha='right')
#     plt.xlabel("Model Snapshots")
#     plt.ylabel("Aggregated IoU")
#     plt.title(f"Aggregated Proper Heads IoU Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
    
#     iou_filename = os.path.join(iou_dir, f"{relation_name}_heads_iou.pdf")
#     #plt.savefig(iou_filename, dpi=500, bbox_inches='tight', format="pdf")
#     print(f"IoU plot saved to {iou_filename}")
#     plt.show()
#     plt.close()
    
#     # ------------------------------
#     # Plot and save the Aggregated Head Count figure.
#     # ------------------------------
#     plt.figure(figsize=(20, 6))
#     plt.plot(abbreviated_model_names, general_head_count, label="Proper General Heads Count (Aggregated)", marker='o', linestyle='-')
#     plt.plot(abbreviated_model_names, subject_head_count, label="Proper Subject Heads Count (Aggregated)", marker='s', linestyle='--')
#     plt.plot(abbreviated_model_names, relation_head_count, label="Proper Relation Heads Count (Aggregated)", marker='x', linestyle='-.')
#     plt.plot(abbreviated_model_names, answer_head_count, label="Proper Answer Heads Count (Aggregated)", marker='^', linestyle='-.')

#     plt.xticks(rotation=45, ha='right')
#     plt.xlabel("Model Snapshots")
#     plt.ylabel("Aggregated Count")
#     plt.title(f"Aggregated Proper Heads Count Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
    
#     count_filename = os.path.join(count_dir, f"{relation_name}_heads_count.pdf")
#     #plt.savefig(count_filename, dpi=500, bbox_inches='tight', format="pdf")
#     print(f"Head count plot saved to {count_filename}")
#     plt.show()
#     plt.close()

In [ ]:
relations = [
    "city_in_country", "company_hq", "country_capital_city",
    "food_from_country", "official_language", "plays_sport", "sights_in_city", "books_written", "company_ceo", "movie_directed"
]

In [ ]:
def calculate_consistency_proper_heads_all_facts(heatmaps_dict, relation_name):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    for model_name, model_data in heatmaps_dict.items():
        #if model_name == 'main':
        #    continue

        # Store IoU, intersection, and union results for this model
        model_metrics = {heatmap_type: {'iou': [], 'head_count': [], 'intersection': [], 'union': []}
                         for heatmap_type in main_heatmaps.keys()}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "proper_relation_answer_heads":
                # Iterate over all facts (assume array structure)
                #for fact_idx in range(len(main_heatmaps[heatmap_type][relation_name])):
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]

                # Calculate metrics
                intersection = np.logical_and(main_heatmap, model_heatmap)
                union = np.logical_or(main_heatmap, model_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

            elif heatmap_type == "proper_answer_specific_heads":
                # Iterate over all facts and answers
                #for fact_idx in range(len(main_heatmaps[heatmap_type][relation_name])):
                    #for answer_id, main_specific_heatmap in enumerate(
                    #    main_heatmaps[heatmap_type][relation_name][fact_idx]
                    #~):
                main_specific_heatmap = np.logical_or.reduce(list(main_heatmaps[heatmap_type][relation_name].values())) #main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_specific_heatmap = np.logical_or.reduce(list(model_data[heatmap_type][relation_name].values())) #model_data[heatmap_type][relation_name][fact_idx]#[answer_id]

                # Calculate metrics
                intersection = np.logical_and(main_specific_heatmap, model_specific_heatmap)
                union = np.logical_or(main_specific_heatmap, model_specific_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_specific_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

            else:
                # Handle proper_entity_heads and proper_general_heads
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]

                # Calculate metrics
                intersection = np.logical_and(main_heatmap, model_heatmap)
                union = np.logical_or(main_heatmap, model_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

        # Aggregate results for each heatmap type
        aggregated_metrics = {ht: {
            'iou': np.mean(metrics['iou']),
            'head_count': np.sum(metrics['head_count']),
            'intersection': np.sum(metrics['intersection']),
            'union': np.sum(metrics['union'])
        } for ht, metrics in model_metrics.items()}

        iou_results[model_name] = aggregated_metrics

    return iou_results


# def plot_aggregated_proper_heads_iou(iou_results, relation_name, output_dir):
#     # Prepare data
#     model_names = sorted(
#         iou_results.keys(),
#         key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
#     )
#     general_ious = []
#     entity_ious = []
#     answer_all_ious = []
#     answer_ious = []

#     general_head_count = []
#     entity_head_count = []
#     answer_all_head_count = []
#     answer_head_counts = []

#     # Populate lists for each heatmap type
#     for model_name in model_names[:-1]:
#         general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
#         entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
#         answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
#         answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])

#     for model_name in model_names:
#         general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
#         entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
#         answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
#         answer_head_counts.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])

#     # Create directories if not exist
#     iou_dir = os.path.join(output_dir, "iou")
#     count_dir = os.path.join(output_dir, "count")
#     os.makedirs(iou_dir, exist_ok=True)
#     os.makedirs(count_dir, exist_ok=True)

#     # Plot and save IoU
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_names[:-1], general_ious, label="Proper General Heads IoU (Aggregated)", marker='o', linestyle='-')
#     plt.plot(model_names[:-1], entity_ious, label="Proper Entity Heads IoU (Aggregated)", marker='s', linestyle='--')
#     plt.plot(model_names[:-1], answer_all_ious, label="Proper Relation Answer Heads IoU (Aggregated)", marker='x', linestyle='-.')
#     plt.plot(model_names[:-1], answer_ious, label="Proper Answer Specific Heads IoU (Aggregated)", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_names[:-1])), labels=model_names[:-1], rotation=45, ha='right')
#     plt.xlabel("Model Steps")
#     plt.ylabel("Aggregated IoU")
#     plt.title(f"Aggregated Proper Heads IoU Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()
#     #plt.savefig(os.path.join(iou_dir, f"{relation_name}_heads_iou.png"))
#     plt.close()

#     # Plot and save Count
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_names, general_head_count, label="Proper General Heads Count (Aggregated)", marker='o', linestyle='-')
#     plt.plot(model_names, entity_head_count, label="Proper Entity Heads Count (Aggregated)", marker='s', linestyle='--')
#     plt.plot(model_names, answer_all_head_count, label="Proper Relation Answer Heads Count (Aggregated)", marker='x', linestyle='-.')
#     plt.plot(model_names, answer_head_counts, label="Proper Answer Specific Heads Count (Aggregated)", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
#     plt.xlabel("Model Steps")
#     plt.ylabel("Aggregated Count")
#     plt.title(f"Aggregated Proper Heads Count Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     #plt.savefig(os.path.join(count_dir, f"{relation_name}_heads_count.png"))
#     plt.show()
#     plt.close()

In [ ]:
# def calculate_consistency_proper_heads_all_facts(heatmaps_dict, relation_name):
#     iou_results = {}

#     # Retrieve the 'main' model's proper heads heatmaps
#     main_heatmaps = heatmaps_dict['main']

#     for model_name, model_data in heatmaps_dict.items():
#         # Initialize metrics storage for each heatmap type for this model.
#         model_metrics = {
#             ht: {'iou': [], 'head_count': [], 'intersection': [], 'union': []}
#             for ht in main_heatmaps.keys()
#         }

#         for heatmap_type in main_heatmaps.keys():
#             if heatmap_type == "proper_relation_answer_heads":
#                 # For relation answer heads, index by relation_name.
#                 main_heatmap = main_heatmaps[heatmap_type][relation_name]
#                 model_heatmap = model_data[heatmap_type][relation_name]

#                 intersection = np.logical_and(main_heatmap, model_heatmap)
#                 union = np.logical_or(main_heatmap, model_heatmap)

#                 area_of_intersection = np.sum(intersection)
#                 area_of_union = np.sum(union)
#                 iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

#                 model_metrics[heatmap_type]['iou'].append(iou)
#                 model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
#                 model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
#                 model_metrics[heatmap_type]['union'].append(area_of_union)

#             elif heatmap_type == "proper_answer_specific_heads":
#                 # For answer-specific heads, combine all fact-specific maps for the relation.
#                 main_specific_heatmap = np.logical_or.reduce(
#                     list(main_heatmaps[heatmap_type][relation_name].values())
#                 )
#                 model_specific_heatmap = np.logical_or.reduce(
#                     list(model_data[heatmap_type][relation_name].values())
#                 )

#                 intersection = np.logical_and(main_specific_heatmap, model_specific_heatmap)
#                 union = np.logical_or(main_specific_heatmap, model_specific_heatmap)

#                 area_of_intersection = np.sum(intersection)
#                 area_of_union = np.sum(union)
#                 iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

#                 model_metrics[heatmap_type]['iou'].append(iou)
#                 model_metrics[heatmap_type]['head_count'].append(model_specific_heatmap.sum())
#                 model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
#                 model_metrics[heatmap_type]['union'].append(area_of_union)

#             else:
#                 # For proper_entity_heads and proper_general_heads, index per relation.
#                 main_heatmap = main_heatmaps[heatmap_type][relation_name]
#                 model_heatmap = model_data[heatmap_type][relation_name]

#                 intersection = np.logical_and(main_heatmap, model_heatmap)
#                 union = np.logical_or(main_heatmap, model_heatmap)

#                 area_of_intersection = np.sum(intersection)
#                 area_of_union = np.sum(union)
#                 iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

#                 model_metrics[heatmap_type]['iou'].append(iou)
#                 model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
#                 model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
#                 model_metrics[heatmap_type]['union'].append(area_of_union)

#         # Aggregate metrics for each heatmap type.
#         aggregated_metrics = {
#             ht: {
#                 'iou': np.mean(metrics['iou']),
#                 'head_count': np.sum(metrics['head_count']),
#                 'intersection': np.sum(metrics['intersection']),
#                 'union': np.sum(metrics['union'])
#             }
#             for ht, metrics in model_metrics.items()
#         }

#         iou_results[model_name] = aggregated_metrics

#     return iou_results

# def plot_aggregated_proper_heads_iou(iou_results, relation_name, output_dir):
#     # Prepare data
#     model_names = sorted(
#         iou_results.keys(),
#         key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
#     )
#     general_ious = []
#     entity_ious = []
#     answer_all_ious = []
#     answer_ious = []

#     general_head_count = []
#     entity_head_count = []
#     answer_all_head_count = []
#     answer_head_counts = []

#     # Populate lists for each heatmap type
#     for model_name in model_names[:-1]:
#         general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
#         entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
#         answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
#         answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])

#     for model_name in model_names:
#         general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
#         entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
#         answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
#         answer_head_counts.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])

#     # Create directories if not exist
#     iou_dir = os.path.join(output_dir, "iou")
#     count_dir = os.path.join(output_dir, "count")
#     os.makedirs(iou_dir, exist_ok=True)
#     os.makedirs(count_dir, exist_ok=True)

#     # Plot and save IoU
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_names[:-1], general_ious, label="Proper General Heads IoU (Aggregated)", marker='o', linestyle='-')
#     plt.plot(model_names[:-1], entity_ious, label="Proper Entity Heads IoU (Aggregated)", marker='s', linestyle='--')
#     plt.plot(model_names[:-1], answer_all_ious, label="Proper Relation Answer Heads IoU (Aggregated)", marker='x', linestyle='-.')
#     plt.plot(model_names[:-1], answer_ious, label="Proper Answer Specific Heads IoU (Aggregated)", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_names[:-1])), labels=model_names[:-1], rotation=45, ha='right')
#     plt.xlabel("Model Steps")
#     plt.ylabel("Aggregated IoU")
#     plt.title(f"Aggregated Proper Heads IoU Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()
#     #plt.savefig(os.path.join(iou_dir, f"{relation_name}_heads_iou.png"))
#     plt.close()

#     # Plot and save Count
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_names, general_head_count, label="Proper General Heads Count (Aggregated)", marker='o', linestyle='-')
#     plt.plot(model_names, entity_head_count, label="Proper Entity Heads Count (Aggregated)", marker='s', linestyle='--')
#     plt.plot(model_names, answer_all_head_count, label="Proper Relation Answer Heads Count (Aggregated)", marker='x', linestyle='-.')
#     plt.plot(model_names, answer_head_counts, label="Proper Answer Specific Heads Count (Aggregated)", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
#     plt.xlabel("Model Steps")
#     plt.ylabel("Aggregated Count")
#     plt.title(f"Aggregated Proper Heads Count Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     #plt.savefig(os.path.join(count_dir, f"{relation_name}_heads_count.png"))
#     plt.show()
#     plt.close()

In [ ]:
# def calculate_consistency_proper_heads_all_facts(heatmaps_dict, relation_name):
#     """
#     Calculates consistency metrics (IoU, head counts, intersection, union) for each proper head type 
#     for the given relation across all models. Uses the new proper head keys:
#       - proper_general_heads
#       - proper_subject_heads
#       - proper_relation_heads
#       - proper_answer_heads

#     Assumes heatmaps_dict is structured as:
#       {
#         model_name: {
#             relation_name: {
#                 'proper_general_heads': np.array(...),
#                 'proper_subject_heads': np.array(...),
#                 'proper_relation_heads': np.array(...),
#                 'proper_answer_heads': np.array(...)
#             },
#             ...
#         },
#         ...
#       }

#     Args:
#         heatmaps_dict (dict): Dictionary of proper head heatmaps for each model.
#         relation_name (str): The relation for which metrics are computed.

#     Returns:
#         dict: Dictionary of aggregated metrics for each model and head type.
#               Example:
#               {
#                   model_name: {
#                       'proper_general_heads': {'iou': ..., 'head_count': ..., 'intersection': ..., 'union': ...},
#                       'proper_subject_heads': {...},
#                       'proper_relation_heads': {...},
#                       'proper_answer_heads': {...}
#                   },
#                   ...
#               }
#     """
#     iou_results = {}
#     # Retrieve the 'main' model's proper heads for the given relation.
#     main_relation_heads = heatmaps_dict['main'][relation_name]

#     for model_name, model_data in heatmaps_dict.items():
#         model_relation_heads = model_data[relation_name]
#         model_metrics = {}
        
#         for head_type in ["proper_general_heads", "proper_subject_heads", "proper_relation_heads", "proper_answer_heads"]:
#             main_heatmap = main_relation_heads[head_type]
#             model_heatmap = model_relation_heads[head_type]
            
#             intersection = np.logical_and(main_heatmap, model_heatmap)
#             union = np.logical_or(main_heatmap, model_heatmap)
            
#             area_of_intersection = np.sum(intersection)
#             area_of_union = np.sum(union)
#             iou = area_of_intersection / area_of_union if area_of_union > 0 else 0
            
#             model_metrics[head_type] = {
#                 'iou': iou,
#                 'head_count': np.sum(model_heatmap),
#                 'intersection': area_of_intersection,
#                 'union': area_of_union
#             }
        
#         iou_results[model_name] = model_metrics

#     return iou_results

In [ ]:
relation_names = models_data['main'].keys()
output_dir = "final_figures2"
for relation_name in relations:
    # Extract the sentences for all facts (optional, for reference)
    sentences = [models_data['main'][relation_name][fact_idx]['sentence'] for fact_idx in range(len(models_data['main'][relation_name]))]

    # Step 1: Calculate proper heads
    proper_heads = calculate_proper_heads(heatmaps_dict)

    # Step 2: Calculate overlap (IoU) across all facts for the relation
    overlap = calculate_consistency_proper_heads_all_facts(proper_heads, relation_name)

    # Step 3: Plot aggregated IoU across all facts for the relation
    plot_aggregated_proper_heads_iou(overlap, relation_name, output_dir)

In [ ]:
relation_names = models_data['main'].keys()
output_dir = "final_figures2"
for relation_name in relations:
    # Extract the sentences for all facts (optional, for reference)
    sentences = [models_data['main'][relation_name][fact_idx]['sentence'] for fact_idx in range(len(models_data['main'][relation_name]))]

    # Step 1: Calculate proper heads
    proper_heads = calculate_proper_heads(heatmaps_dict)

    # Step 2: Calculate overlap (IoU) across all facts for the relation
    overlap = calculate_consistency_proper_heads_all_facts(proper_heads, relation_name)

    # Step 3: Plot aggregated IoU across all facts for the relation
    plot_aggregated_proper_heads_iou(overlap, relation_name, output_dir)

In [ ]:
def compute_head_paths(snapshots):
    """
    Given a dictionary of snapshots, compute for each head (32x32) a list of roles
    over the snapshots.
    
    Parameters:
        snapshots (dict): Keys are snapshot names (e.g., 'step5000-tokens20B' or 'main') and values
                          are dictionaries with keys 'general_heatmap', 'entity_heatmap',
                          'relation_answer_heatmap', 'answer_specific_heatmap' (each a 32x32
                          Boolean numpy array).
    
    Returns:
        paths (dict): A dictionary whose keys are (i, j) tuples (the head positions) and whose
                      values are lists (one per snapshot, in order) giving the role of that head.
                      The role is one of 'general', 'entity', 'relation_answer', 'answer_specific',
                      or 'deactivated'.
    """
    # Separate the 'main' snapshot from the others.
    non_main_keys = [k for k in snapshots if k != "main"]
    # Sort non-main keys based on the numeric value after "step"
    sorted_keys = sorted(non_main_keys, key=lambda k: int(k.split('-')[0].replace('step','')))
    
    # If there is a 'main' snapshot, add it as the last element.
    if "main" in snapshots:
        sorted_keys.append("main")
    
    paths = {}  # to store the path (list of roles) for each head
    # There are 32x32 = 1024 heads.
    for i in range(32):
        for j in range(32):
            head_path = []  # will store the role for this head at each snapshot
            for snap_key in sorted_keys:
                snap = snapshots[snap_key]
                # Determine the role for head (i, j) in this snapshot.
                # Adjust the order if needed. If more than one heatmap is True, this code
                # picks the first one in the order below.
                if snap['proper_general_heads'][i, j]:
                    role = 'general'
                elif snap['proper_entity_heads'][i, j]:
                    role = 'entity'
                elif snap['proper_relation_answer_heads'][i, j]:
                    role = 'relation_answer'
                elif snap['proper_answer_specific_heads'][i, j]:
                    role = 'answer_specific'
                else:
                    role = 'deactivated'
                head_path.append(role)
            paths[(i, j)] = head_path

    return paths


In [ ]:
heatmaps_dict_agg

In [ ]:
proper_heads_agg

In [ ]:
proper_heads

In [ ]:
from collections import Counter
import numpy as np

def compute_head_paths(snapshots, relation_name=None):
    """
    Given a dictionary of snapshots, compute for each head (32x32) a list of roles
    over the snapshots.
    
    Parameters:
        snapshots (dict): Keys are snapshot names (e.g., 'step5000-tokens20B' or 'main') and values
                          are dictionaries with keys:
                            - 'proper_general_heads'
                            - 'proper_entity_heads'
                            - 'proper_relation_answer_heads'
                            - 'proper_answer_specific_heads'
                          For the relation‑specific keys ('proper_relation_answer_heads' and 
                          'proper_answer_specific_heads'), the value can be either:
                            • a 32x32 Boolean NumPy array, or
                            • a dictionary mapping relation names to 32x32 Boolean arrays. 
                          (In the case of proper_answer_specific_heads, the value for a given relation
                           may itself be a dictionary mapping to several Boolean arrays.)
        relation_name (str, optional): If provided and the relation‑specific maps are dictionaries,
                          the specified relation's aggregated map is used. For proper_answer_specific_heads,
                          if the relation mapping is a dictionary, its values will be aggregated with
                          np.logical_or.reduce.
    
    Returns:
        paths (dict): A dictionary whose keys are (i, j) tuples (the head positions) and whose
                      values are lists (one per snapshot, in order) giving the role of that head.
                      The role is one of 'general', 'entity', 'relation_answer', 'answer_specific',
                      or 'deactivated'.
    """
    # Sort snapshot keys: first, all non-"main" snapshots (sorted by the numeric part in the key)
    # and then the "main" snapshot at the end.
    non_main_keys = [k for k in snapshots if k != "main"]
    sorted_keys = sorted(non_main_keys, key=lambda k: int(k.split('-')[0].replace('step','')))
    if "main" in snapshots:
        sorted_keys.append("main")
    
    paths = {}  # to store the path (list of roles) for each head (i, j)
    
    # There are 32x32 = 1024 heads.
    for i in range(32):
        for j in range(32):
            head_path = []  # will store the role for this head at each snapshot
            for snap_key in sorted_keys:
                snap = snapshots[snap_key]
                
                # Retrieve the aggregated maps.
                general_map = snap['proper_general_heads']
                entity_map = snap['proper_entity_heads']
                
                # For proper_relation_answer_heads: if a relation is specified and the map is a dictionary,
                # select that relation's 32x32 map.
                if relation_name is not None and isinstance(snap['proper_relation_answer_heads'], dict):
                    relation_answer_map = snap['proper_relation_answer_heads'][relation_name]
                else:
                    relation_answer_map = snap['proper_relation_answer_heads']
                    
                # For proper_answer_specific_heads: if a relation is specified and the value is a dictionary,
                # check if it itself is a dictionary. If so, aggregate its values.
                if relation_name is not None and isinstance(snap['proper_answer_specific_heads'], dict):
                    specific_val = snap['proper_answer_specific_heads'][relation_name]
                    if isinstance(specific_val, dict):
                        answer_specific_map = np.logical_or.reduce(list(specific_val.values()))
                    else:
                        answer_specific_map = specific_val
                else:
                    answer_specific_map = snap['proper_answer_specific_heads']
                
                # Determine the role of head (i, j) in this snapshot.
                # If more than one heatmap is True, the first in the order below is chosen.
                if general_map[i, j]:
                    role = 'general'
                elif entity_map[i, j]:
                    role = 'entity'
                elif relation_answer_map[i, j]:
                    role = 'relation_answer'
                elif answer_specific_map[i, j]:
                    role = 'answer_specific'
                else:
                    role = 'deactivated'
                head_path.append(role)
            paths[(i, j)] = head_path

    return paths


def count_paths(head_paths):
    """
    Given the head_paths dictionary, count how many heads follow each unique path.
    
    Parameters:
        head_paths (dict): Keys are head positions (i,j) and values are lists (paths).
        
    Returns:
        path_counter (Counter): A collections.Counter object mapping each unique path (as a tuple)
                                  to the number of heads that follow that path.
    """
    # Convert each head's path (a list) to a tuple so it can be used as a key in a Counter.
    path_counter = Counter(tuple(path) for path in head_paths.values())
    return path_counter


def group_paths(head_paths):
    """
    Given the head_paths dictionary, group head positions by their unique path.
    
    Parameters:
        head_paths (dict): Keys are head positions (i,j) and values are lists (paths).
        
    Returns:
        groups (dict): A dictionary mapping each unique path (as a tuple) to a list of head positions
                       that share that path.
    """
    groups = {}
    for head, path in head_paths.items():
        path_key = tuple(path)  # convert list to tuple for hashing
        groups.setdefault(path_key, []).append(head)
    return groups

def compute_overall_transition_probabilities(head_paths, roles):
    """
    Compute an overall transition probability matrix by summing over all consecutive revision pairs.
    
    Returns:
        P: A numpy array where P[i, j] is the probability of transitioning from role i to role j.
    """
    num_roles = len(roles)
    role_to_idx = {role: i for i, role in enumerate(roles)}
    transition_counts = np.zeros((num_roles, num_roles), dtype=float)
    total_from_counts = np.zeros(num_roles, dtype=int)
    
    for path in head_paths.values():
        for i in range(len(path) - 1):
            src = path[i]
            tgt = path[i + 1]
            s_idx = role_to_idx[src]
            t_idx = role_to_idx[tgt]
            transition_counts[s_idx, t_idx] += 1
            total_from_counts[s_idx] += 1
            
    P = np.zeros_like(transition_counts)
    for i in range(num_roles):
        if total_from_counts[i] > 0:
            P[i, :] = transition_counts[i, :] / total_from_counts[i]
    return P


In [ ]:
relations = [
    "city_in_country", "company_hq", "country_capital_city",
    "food_from_country", "official_language", "plays_sport", "sights_in_city",
    "books_written", "company_ceo", "movie_directed"
]

# List of roles
roles = ['general', 'entity', 'relation_answer', 'answer_specific', 'deactivated']

# For demonstration, assume there are 5 revisions (replace with 40 or your actual number).
n_revisions = 40

# Iterate over each relation.
for relation in relations:
    # Compute head paths for the current relation.
    head_paths = compute_head_paths(proper_heads, relation_name=relation)
    
    # Count the unique paths.
    path_counter = count_paths(head_paths)
    groups = group_paths(head_paths)
    
    # # Define a filename for the current relation.
    # filename = f"final_figures2/path_output/path_counts_{relation}.txt"
    # filename_id = f"final_figures2/path_output/path_id_{relation}.txt"
    
    # # Open the file and write the results.
    # with open(filename, "w") as f:
    #     # Write a header (optional)
    #     f.write(f"Unique path counts for relation: {relation}\n")
    #     f.write("-" * 40 + "\n")
        
    #     # Write the path counts sorted by frequency (most common first).
    #     for path, count in path_counter.most_common():
    #         f.write(f"Path {path} appears in {count} heads.\n")
    
    # with open(filename_id, "w") as f:
    #     # Write a header (optional)
    #     f.write(f"Grouped head positions for relation: {relation}\n")
    #     f.write("-" * 40 + "\n")
    #     for path, heads in groups.items():
    #         f.write(f"Path {path} is shared by {len(heads)} heads: {heads}\n")
    #     print(f"Grouped output for relation '{relation}' saved to {filename_id}")
    
    
    # print(f"Output for relation '{relation}' saved to {filename}")
    
    # Compute the overall transition probability matrix.
    P = compute_overall_transition_probabilities(head_paths, roles)

    # Print a descriptive table of transitions, including the relation name.
    print(f"Overall Transition Probability Matrix for relation: {relation} (each cell shows P[From → To]):")
    header = "From \\ To\t" + "\t".join(roles)
    print(header)
    for i, role in enumerate(roles):
        row = role + "\t\t" + "\t".join(f"{P[i, j]:.2f}" for j in range(len(roles)))
        print(row)

    # Create the heatmap.
    plt.figure(figsize=(8, 6))
    sns.heatmap(P, annot=True, fmt=".2f", cmap="YlGnBu", xticklabels=roles, yticklabels=roles)
    plt.xlabel("Target Role")
    plt.ylabel("Source Role")
    plt.title(f"Markov Chain Transition Probability Heatmap - {relation}")

    # Save the plot as a PNG file.
    filename = f"final_figures2/path_output/transition_probability_heatmap_{relation}.pdf"
    plt.savefig(filename, dpi=500, bbox_inches='tight', format="pdf")
    print(f"Plot saved as {filename}")

    plt.show()

In [ ]:
import math
import matplotlib.pyplot as plt
import seaborn as sns

# List of relations to process.
relations = [
    "city_in_country", "company_hq", "country_capital_city",
    "food_from_country", "official_language", "plays_sport", "sights_in_city",
    "books_written", "company_ceo", "movie_directed"
]

# List of roles.
roles = ['general', 'entity', 'relation_answer', 'answer_specific', 'deactivated']

# Set up the grid dimensions.
n_relations = len(relations)
n_cols = 2  # You can adjust the number of columns.
n_rows = math.ceil(n_relations / n_cols)

# Create a figure with subplots.
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
axes = axes.flatten()  # Flatten to easily iterate over them.

# Iterate over each relation.
for idx, relation in enumerate(relations):
    # Compute head paths for the current relation.
    head_paths = compute_head_paths(proper_heads, relation_name=relation)
    
    # Compute the overall transition probability matrix.
    P = compute_overall_transition_probabilities(head_paths, roles)
    
    # Plot the heatmap for this relation on the corresponding subplot.
    ax = axes[idx]
    sns.heatmap(P, annot=True, fmt=".2f", cmap="YlGnBu",
                xticklabels=roles, yticklabels=roles, ax=ax)
    ax.set_xlabel("Target Role")
    ax.set_ylabel("Source Role")
    ax.set_title(f"{relation}")

# Hide any unused subplots (if n_rows*n_cols > number of relations).
for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

# Save the combined figure.
combined_filename = "final_figures2/path_output/transition_probability_heatmap_combined2.pdf"
plt.savefig(combined_filename, dpi=500, bbox_inches='tight', format="pdf")
print(f"Combined plot saved as {combined_filename}")

plt.show()

In [ ]:
from collections import Counter

path_counts = count_paths(head_paths)

# Sort the counts from highest to lowest using most_common():
sorted_path_counts = path_counts.most_common()

print("Unique path counts sorted by frequency (most common first):")
for path, count in sorted_path_counts:
    print(f"Path {path} appears in {count} heads.")

In [ ]:
# # Option 2: Group the heads by their unique path and sort the groups by their count.
groups = group_paths(head_paths)
sorted_groups = sorted(groups.items(), key=lambda item: len(item[1]), reverse=True)

print("\nGroups of heads by identical path, sorted by count:")
for path, heads in sorted_groups:
    print(f"Path {path} has {len(heads)} heads: {heads}")

In [ ]:
import numpy as np
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import pandas as pd
import scipy.cluster.hierarchy as sch


# List of roles
roles = ['general', 'entity', 'relation_answer', 'answer_specific', 'deactivated']

# For demonstration, assume there are 5 revisions (replace with 40 or your actual number).
n_revisions = 40


# Compute the overall transition probability matrix.
P = compute_overall_transition_probabilities(head_paths, roles)

# Print a descriptive table of transitions.
print("Overall Transition Probability Matrix (each cell shows P[From → To]):")
header = "From \\ To\t" + "\t".join(roles)
print(header)
for i, role in enumerate(roles):
    row = role + "\t\t" + "\t".join(f"{P[i, j]:.2f}" for j in range(len(roles)))
    print(row)

plt.figure(figsize=(8, 6))
sns.heatmap(P, annot=True, fmt=".2f", cmap="YlGnBu", xticklabels=roles, yticklabels=roles)
plt.xlabel("Target Role")
plt.ylabel("Source Role")
plt.title("Markov Chain Transition Probability Heatmap")
combined_filename = "final_figures2/path_output/transition_probability_heatmap_aggregated.pdf"
plt.savefig(combined_filename, dpi=500, bbox_inches='tight', format="pdf")
plt.show()

In [ ]:
def compute_heatmaps_agg(models_data, sorted_models, theta=0.1):
    """
    Computes aggregated heatmaps for a set of models.
    
    - The overall general and entity heatmaps are aggregated over all relations.
    - The relation-answer and answer-specific heatmaps are computed separately 
      for the two relation groups (LOC and NAME) and then aggregated over all relations
      within each group.
    
    Args:
        models_data (dict): Dictionary where each key is a model name and the value is a dict
                            mapping relation names to lists of entries.
        sorted_models (list): List of model names in the desired order.
        theta (float): Threshold for binarizing the heatmaps.
        
    Returns:
        tuple: Two dictionaries:
            - heatmaps_dict_LOC: For each model, contains the overall general and entity heatmaps (shared)
              plus an aggregated relation-answer heatmap and aggregated answer-specific heatmap for LOC relations.
            - heatmaps_dict_NAME: Similarly, for NAME relations.
    """
    # Define the two relation groups (in lower-case)
    loc_relations = [
        "city_in_country", "company_hq", "country_capital_city",
        "food_from_country", "official_language", "plays_sport", "sights_in_city", "books_written", "company_ceo", "movie_directed"
    ]
    name_relations = [
        "books_written", "company_ceo", "movie_directed"
    ]
    
    heatmaps_dict_LOC = {}
    heatmaps_dict_NAME = {}
    
    for model_name in sorted_models:
        relations_data_model = models_data[model_name]
        print(model_name)
        
        # --- Overall (general and entity) heatmaps: computed over all relations ---
        g_total = 0
        general_heatmap = np.zeros((32, 32), dtype=float)
        e_total = 0
        entity_heatmap = np.zeros((32, 32), dtype=float)
        
        # --- Aggregators for relation-answer and answer-specific heatmaps per group ---
        # For LOC
        aggregated_relation_answer_heatmap_LOC = np.zeros((32, 32), dtype=bool)
        aggregated_answer_specific_heatmap_LOC = np.zeros((32, 32), dtype=bool)
        # For NAME
        aggregated_relation_answer_heatmap_NAME = np.zeros((32, 32), dtype=bool)
        aggregated_answer_specific_heatmap_NAME = np.zeros((32, 32), dtype=bool)
        
        # Process each relation in the model's data.
        for relation, entries in relations_data_model.items():
            # Process only if the relation is in one of our groups.
            if relation not in loc_relations + name_relations:
                continue
            
            # Prepare accumulators for the current relation.
            relation_answer_heatmap = np.zeros((32, 32), dtype=float)
            relation_answer_total = 0
            # We'll also store answer-specific heatmaps in a list so that we can aggregate them.
            answer_specific_list = []
            
            for idx, entry in enumerate(entries):
                # Ensure subject token span is defined.
                if entry["subj_token_span"] is None:
                    entry["subj_token_span"] = np.arange(len(entry["subject_tokens"])).tolist()
                    
                # --- Update overall general heatmap (averaged over all entries) ---
                general_heatmap += np.mean(entry['attnheads_contribution_maps'], axis=0)
                g_total += 1
                
                # --- Update overall entity heatmap for subject tokens ---
                one_data_map = np.zeros((32, 32), dtype=float)
                for t in entry["subj_token_span"]:
                    one_data_map += entry['attnheads_contribution_maps'][t]
                entity_heatmap += one_data_map
                e_total += len(entry["subj_token_span"])
                
                # --- Update overall entity and relation-answer heatmaps for answer tokens ---
                one_data_map = np.zeros((32, 32), dtype=float)
                for t in entry["answer_token_span"]:
                    one_data_map += entry['attnheads_contribution_maps'][t - 1]
                entity_heatmap += one_data_map
                e_total += len(entry["answer_token_span"])
                relation_answer_heatmap += one_data_map
                relation_answer_total += len(entry["answer_token_span"])
                
                # --- Compute answer-specific heatmap for this entry ---
                answer_specific_heatmap = one_data_map / len(entry["answer_token_span"])
                answer_specific_list.append(answer_specific_heatmap > theta)
            
            # Normalize the relation-answer heatmap for the current relation.
            if relation_answer_total > 0:
                relation_answer_heatmap /= relation_answer_total
            else:
                relation_answer_heatmap = np.zeros((32, 32), dtype=float)
            
            # Threshold the relation-answer heatmap.
            relation_answer_thresh = relation_answer_heatmap > theta
            
            # Aggregate this relation's heatmaps into the corresponding group accumulator.
            if relation in loc_relations:
                aggregated_relation_answer_heatmap_LOC = np.logical_or(
                    aggregated_relation_answer_heatmap_LOC, relation_answer_thresh)
                # Aggregate each answer-specific heatmap.
                for heat in answer_specific_list:
                    aggregated_answer_specific_heatmap_LOC = np.logical_or(
                        aggregated_answer_specific_heatmap_LOC, heat)
            elif relation in name_relations:
                aggregated_relation_answer_heatmap_NAME = np.logical_or(
                    aggregated_relation_answer_heatmap_NAME, relation_answer_thresh)
                for heat in answer_specific_list:
                    aggregated_answer_specific_heatmap_NAME = np.logical_or(
                        aggregated_answer_specific_heatmap_NAME, heat)
        
        # Normalize overall general and entity heatmaps.
        if g_total > 0:
            general_heatmap /= g_total
        if e_total > 0:
            entity_heatmap /= e_total
        
        # Apply thresholding to overall heatmaps.
        thresholded_general = general_heatmap > theta
        thresholded_entity = entity_heatmap > theta
        
        # Store the results for the current model.
        heatmaps_dict_LOC[model_name] = {
            'general_heatmap': thresholded_general,
            'entity_heatmap': thresholded_entity,
            'relation_answer_heatmaps': aggregated_relation_answer_heatmap_LOC,
            'answer_specific_heatmaps': aggregated_answer_specific_heatmap_LOC
        }
        heatmaps_dict_NAME[model_name] = {
            'general_heatmap': thresholded_general,
            'entity_heatmap': thresholded_entity,
            'relation_answer_heatmaps': aggregated_relation_answer_heatmap_NAME,
            'answer_specific_heatmaps': aggregated_answer_specific_heatmap_NAME
        }
    
    return heatmaps_dict_LOC, heatmaps_dict_NAME

heatmaps_dict_agg, _ = compute_heatmaps_agg(models_data, sorted_models, theta=0.1)

In [ ]:
# List of relations you want to process.
relations = [
    "city_in_country", "company_hq", "country_capital_city",
    "food_from_country", "official_language", "plays_sport", "sights_in_city", "books_written", "company_ceo", "movie_directed"
]

for relation in relations:
    # Compute head paths for the current relation.
    head_paths = compute_head_paths(proper_heads, relation_name=relation)
    
    # Count the unique paths.
    path_counter = count_paths(head_paths)
    
    # Group head positions by path.
    groups = group_paths(head_paths)
    
    # Save path counts to a text file.
    filename_counts = f"path_counts_{relation}.txt"
    with open(filename_counts, "w") as f:
        f.write(f"Unique path counts for relation: {relation}\n")
        f.write("-" * 40 + "\n")
        for path, count in path_counter.most_common():
            f.write(f"Path {path} appears in {count} heads.\n")
    print(f"Output for relation '{relation}' saved to {filename_counts}")
    
    # Save groups to a separate text file.
    filename_groups = f"group_paths_{relation}.txt"
    with open(filename_groups, "w") as f:
        f.write(f"Grouped head positions for relation: {relation}\n")
        f.write("-" * 40 + "\n")
        for path, heads in groups.items():
            f.write(f"Path {path} is shared by {len(heads)} heads: {heads}\n")
    print(f"Grouped output for relation '{relation}' saved to {filename_groups}")

In [ ]:
import pandas as pd

# List of relations you want to process.
relations = [
    "city_in_country", "company_hq", "country_capital_city",
    "food_from_country", "official_language", "plays_sport", "sights_in_city", 
    "books_written", "company_ceo", "movie_directed"
]

for relation in relations:
    # Compute head paths for the current relation.
    head_paths = compute_head_paths(proper_heads, relation_name=relation)
    
    # Count the unique paths.
    path_counter = count_paths(head_paths)
    
    # Group head positions by path.
    groups = group_paths(head_paths)
    
    # Convert path_counter to a DataFrame.
    # Each row will contain a unique path (as a tuple converted to string) and its count.
    df_counts = pd.DataFrame(
        [(str(path), count) for path, count in path_counter.items()],
        columns=['Path', 'Count']
    )
    df_counts = df_counts.sort_values(by='Count', ascending=False)
    counts_filename = f"path_counts_{relation}.csv"
    df_counts.to_csv(counts_filename, index=False)
    print(f"Path counts for relation '{relation}' saved to {counts_filename}")
    
    # Convert groups to a DataFrame.
    # Each row will have the unique path, the number of heads, and a string representation of the heads.
    df_groups = pd.DataFrame(
        [(str(path), len(heads), str(heads)) for path, heads in groups.items()],
        columns=['Path', 'Number of Heads', 'Heads']
    )
    groups_filename = f"group_paths_{relation}.csv"
    df_groups.to_csv(groups_filename, index=False)
    print(f"Grouped paths for relation '{relation}' saved to {groups_filename}")

In [ ]:
import pandas as pd

# List of relations that were processed.
relations = [
    "city_in_country", "company_hq", "country_capital_city",
    "food_from_country", "official_language", "plays_sport", 
    "sights_in_city", "books_written", "company_ceo", "movie_directed"
]

# Create empty lists to hold DataFrames for counts and groups.
counts_dfs = []
groups_dfs = []

# Loop over relations and read the corresponding CSV files.
for relation in relations:
    counts_filename = f"path_counts_{relation}.csv"
    groups_filename = f"group_paths_{relation}.csv"
    
    # Read the CSV files.
    df_counts = pd.read_csv(counts_filename)
    df_groups = pd.read_csv(groups_filename)
    
    # Add a column for the relation to allow grouping later.
    df_counts['Relation'] = relation
    df_groups['Relation'] = relation
    
    counts_dfs.append(df_counts)
    groups_dfs.append(df_groups)

# Combine all DataFrames into one for overall analysis.
all_counts = pd.concat(counts_dfs, ignore_index=True)
all_groupsb = pd.concat(groups_dfs, ignore_index=True)

# Example Analysis 1: Most common paths across all relations.
most_common_paths = (all_counts.groupby('Path')['Count']
                     .sum()
                     .reset_index()
                     .sort_values(by='Count', ascending=False))
print("Most common paths across all relations:")
print(most_common_paths.head(10))

# Example Analysis 2: Most common paths per relation.
for relation in relations:
    df_relation = all_counts[all_counts['Relation'] == relation]
    df_relation = df_relation.sort_values(by='Count', ascending=False)
    print(f"\nMost common paths for relation '{relation}':")
    print(df_relation.head(10))

# Example Analysis 3: How many unique paths are observed per relation.
unique_paths_per_relation = all_counts.groupby('Relation')['Path'].nunique().reset_index()
unique_paths_per_relation.rename(columns={'Path': 'Unique Paths'}, inplace=True)
print("\nUnique paths per relation:")
print(unique_paths_per_relation)

# Example Analysis 4: For the grouped paths, analyze average number of heads per path per relation.
avg_heads_per_path = (all_groups.groupby('Relation')['Number of Heads']
                      .mean()
                      .reset_index()
                      .rename(columns={'Number of Heads': 'Avg Heads per Path'}))
print("\nAverage number of heads per path per relation:")
print(avg_heads_per_path)

In [ ]:
n=5
all_groups["Path"]
filtered_df = all_groups[all_groups['Path'].apply(lambda x: 'answer_specific' in x[-n:])]


In [ ]:
import ast

n = 5  # change this to how many elements from the end you want to check

def has_answer_specific_in_last_n(x):
    # Convert string representation to a tuple if needed
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except Exception as e:
            # Optionally handle the error or return False if conversion fails
            return False
    # Now x is expected to be a tuple or list, slice the last n elements and check
    return 'answer_specific' in x[-n:]

filtered_df = all_groups[all_groups['Path'].apply(has_answer_specific_in_last_n)]

In [ ]:
filtered_df["Path"].values

In [ ]:
def categorize_stability(x):
    # Convert string representation to a tuple if needed
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except Exception as e:
            # If conversion fails, classify as unstable or handle as needed
            return "unstable"
    # Check if all of the last n entries are 'answer_specific'
    if all(item == 'answer_specific' for item in x[-n:]):
        return "stable"
    else:
        return "unstable"

# Apply the function to create a new column 'stability'
filtered_df['stability'] = filtered_df['Path'].apply(categorize_stability)

In [ ]:
filtered_df['stability'].value_counts()

In [ ]:
import ast
import pandas as pd

n = 5  # number of last revisions to examine

def analyze_transition(path, n=5):
    # If the path is stored as a string, convert it to a tuple
    if isinstance(path, str):
        try:
            path = ast.literal_eval(path)
        except Exception:
            return "uncategorized"
    
    # Ensure there are at least n revisions
    if len(path) < n:
        return "uncategorized"
    
    last_n = path[-n:]
    
    # If all revisions are the same, consider the path stable.
    if all(role == last_n[0] for role in last_n):
        return "stable"
    
    # Check for a transition that supports our hypothesis:
    # Look for a transition from answer_specific to general.
    # We want to see that at least one of the earlier entries is answer_specific
    # and that later in the slice the role becomes general.
    try:
        # Find the first occurrence of "general"
        first_general_index = last_n.index("general")
    except ValueError:
        first_general_index = None

    if first_general_index is not None:
        # If any revision before the first "general" is answer_specific, flag as supporting.
        if any(role == "answer_specific" for role in last_n[:first_general_index]):
            return "supporting"
    
    # Alternatively, if the sequence transitions from not being answer_specific to being answer_specific later,
    # that could be taken as contradicting our hypothesis.
    try:
        first_answer_index = last_n.index("answer_specific")
    except ValueError:
        first_answer_index = None

    if first_answer_index is not None and first_answer_index > 0:
        return "contradicting"
    
    return "uncategorized"

# Assuming filtered_df is your DataFrame and the column with paths is "Path",
# apply the function to categorize each entry.
filtered_df['hypothesis_analysis'] = filtered_df['Path'].apply(lambda x: analyze_transition(x, n))

# To see a summary:
analysis_counts = filtered_df['hypothesis_analysis'].value_counts()
print(analysis_counts)

In [ ]:
filtered_df['hypothesis_category'].value_counts()

In [ ]:
import pandas as pd

def classify_last_five(last_five):
    """
    Classify a 5-element tuple of roles (from snapshots S35–S39)
    into one of four categories:
      - "stable": All 5 are "answer-specific".
      - "supporting": e.g. ('answer-specific', 'answer-specific', 'answer-specific', 'general', 'general')
      - "contradicting": e.g. ('deactivated', 'deactivated', 'answer-specific', 'answer-specific', 'answer-specific')
      - "unstable": Everything else.
    """
    if last_five == ("answer_specific",) * 5:
        return "stable"
    elif last_five == ("answer-specific", "answer-specific", "answer-specific", "general", "general"):
        return "supporting"
    elif last_five == ("deactivated", "deactivated", "answer-specific", "answer-specific", "answer-specific"):
        return "contradicting"
    else:
        return "unstable"


def analyze_last_5_paths(groups, n_last=5):
    """
    Analyze the last n_last entries for all paths that contain "answer-specific" somewhere in their full path.
    
    Parameters:
      groups (dict): Dictionary from group_paths, where keys are full path tuples
                     and values are lists of head positions.
      n_last (int): Number of snapshots from the end to consider (default is 5).
    
    Returns:
      pd.DataFrame: A DataFrame with one row per unique path (as defined by its last n_last roles),
                    including the classification category and count of heads following that path.
    """
    data = []
    for full_path, heads in groups.items():
        # Only consider paths that contain "answer-specific" somewhere.
        if "answer_specific" in full_path:
            last_segment = full_path[-n_last:]
            category = classify_last_five(last_segment)
            count = len(heads)
            data.append({
                "Full Path": full_path,
                "Last 5": last_segment,
                "Category": category,
                "Count": count
            })
    df = pd.DataFrame(data)
    return df

relations = [
    "city_in_country", "company_hq", "country_capital_city",
    "food_from_country", "official_language", "plays_sport", "sights_in_city", "books_written", "company_ceo", "movie_directed"
]

for relation in relations:
    # Compute head paths for the current relation.
    head_paths = compute_head_paths(proper_heads, relation_name=relation)
    
    # Count the unique paths.
    path_counter = count_paths(head_paths)
    
    # Group head positions by path.
    groups = group_paths(head_paths)

    df_analysis = analyze_last_5_paths(groups, n_last=5)

    # Show the resulting DataFrame:
    print("Detailed analysis for paths (last 5 snapshots) containing 'answer-specific':")
    print(df_analysis)

    # Summarize counts per category:
    summary = df_analysis.groupby("Category")["Count"].sum().reset_index()
    print("\nSummary of head counts by category:")
    print(summary)

# Optionally, save the analysis to CSV:
# df_analysis.to_csv("last5_paths_analysis.csv", index=False)
# summary.to_csv("last5_paths_summary.csv", index=False)

In [ ]:
relations = [
    "city_in_country", "company_hq", "country_capital_city",
    "food_from_country", "official_language", "plays_sport", "sights_in_city", "books_written", "company_ceo", "movie_directed"
]

for relation in relations:
    # Compute head paths for the current relation.
    head_paths = compute_head_paths(proper_heads, relation_name=relation)
    
    # Count the unique paths.
    path_counter = count_paths(head_paths)
    
    # Group head positions by path.
    groups = group_paths(head_paths)